<a href="https://colab.research.google.com/github/RMoulla/IA_BRGM/blob/main/TP_Analyse_Donn%C3%A9es_BRGM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Ouvrir dans Colab"/></a>


# **Travaux pratiques : analyse des données et modélisation**

In [ ]:
import pandas as pd

dataset = pd.read_csv("customer_churn.csv")
dataset.head(10)


In [ ]:
# Statisitiques descriptives sur les données
dataset.describe()

## Analyse exploratoire des données

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration du style des graphiques
sns.set_theme(style="whitegrid")

# Création de la figure
fig, ax = plt.subplots(figsize=(6, 5))

# Graphique de comptage pour la variable Churn
sns.countplot(data=dataset, x='Churn', palette='viridis', hue='Churn', legend=False, ax=ax)

# Ajout des étiquettes et du titre
plt.title('Distribution de la variable de désabonnement (Churn)')
plt.xlabel('Churn (0 = Non, 1 = Oui)')
plt.ylabel('Nombre de clients')

# Affichage des pourcentages sur les barres
total = len(dataset['Churn'])
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualiser le churn en fonction de l'âge en boxplot

fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=dataset, x='Churn', y='Age', palette='viridis', ax=ax)

In [ ]:
# Visualiser le churn en fonction de total purchase

fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=dataset, x='Churn', y='Total_Purchase', palette='viridis', ax=ax)

In [ ]:
# Visualiser le churn en fonction de Years

fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=dataset, x='Churn', y='Years', palette='viridis', ax=ax)

In [ ]:
# Visualiser le churn en fonction de Num_Sites

fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=dataset, x='Churn', y='Num_Sites', palette='viridis', ax=ax)

In [ ]:
# Visualiser le churn en fonction d'accout manager en countplot

fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(data=dataset, x='Account_Manager', palette='viridis', hue='Churn', ax=ax)

## Modélisation statistique

In [ ]:
# Définir y (churn) et X (Age, Years, Account Manager, Num_Sites et Total Purchase)

y = dataset['Churn']
X = dataset[['Age', 'Years', 'Account_Manager', 'Num_Sites', 'Total_Purchase']]
X.head()

In [ ]:
import statsmodels.api as sm

# Ajouter une constante pour l'intercept (b0)
X_const = sm.add_constant(X)

# Entraîner le modèle de régression logistique
model = sm.Logit(y, X_const).fit()

# Afficher le résumé du modèle
print(model.summary())

## Prédire le churn

In [ ]:
# Partiotionner le jeu de données en train et test

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.head()

In [ ]:
# Ajouter une constante pour l'intercept à l'ensemble d'entraînement
X_train_const = sm.add_constant(X_train)

# Ajuster le modèle de régression logistique sur les données d'entraînement
model_train = sm.Logit(y_train, X_train_const).fit()

# Prédire sur les données de test

X_test_const = sm.add_constant(X_test)
y_pred = model_train.predict(X_test_const)

# Transformer les prédictions en variable binaire

y_pred_binary = [1 if pred >= 0.5 else 0 for pred in y_pred]

# Afficher les prédictions binaires
print(y_pred_binary)

In [ ]:
# Calculer les métriques de performance : accuracy, précision, rappel et f1 score

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_binary)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

## Application de prédiction du Churn avec Gradio

Nous allons installer `gradio` et créer une interface conviviale pour saisir les informations d'un client et obtenir instantanément une prédiction de risque de désabonnement.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import numpy as np

def predict_churn(age, years, account_manager, num_sites, total_purchase):
    # Préparer les données d'entrée avec la constante (intercept) pour le modèle statsmodels
    # Le modèle attend l'ordre : const, Age, Years, Account_Manager, Num_Sites, Total_Purchase
    input_data = [1.0, float(age), float(years), int(account_manager), float(num_sites), float(total_purchase)]

    # Faire la prédiction de probabilité
    probability = model_train.predict(input_data)[0]

    # Déterminer la classe et le message
    class_pred = "Désabonnement (Churn)" if probability >= 0.5 else "Fidèle (Non Churn)"
    color = "#ea4335" if probability >= 0.5 else "#34a853"

    html_output = f"""
    <div style='text-align: center; padding: 20px; border-radius: 10px; background-color: {color}; color: white;'>
        <h2 style='margin: 0;'>Statut prédit : {class_pred}</h2>
        <p style='font-size: 1.2em; margin: 10px 0 0 0;'>Probabilité de churn : {probability:.2%}</p>
    </div>
    """
    return html_output

# Configuration de l'interface Gradio
interface = gr.Interface(
    fn=predict_churn,
    inputs=[
        gr.Slider(minimum=18, maximum=80, value=40, step=1, label="Âge du client"),
        gr.Slider(minimum=1, maximum=15, value=5, step=0.1, label="Nombre d'années d'ancienneté (Years)"),
        gr.Radio(choices=[( "Non", 0), ("Oui", 1)], value=0, label="Dispose d'un Account Manager ?"),
        gr.Slider(minimum=1, maximum=20, value=8, step=1, label="Nombre de sites Web utilisés (Num_Sites)"),
        gr.Number(value=10000, label="Montant total des achats (Total Purchase)")
    ],
    outputs=gr.HTML(label="Résultat de la prédiction"),
    title="🔮 Prédiction du Churn Client",
    description="Saisissez les caractéristiques du client pour évaluer son risque de désabonnement.",
    theme="soft"
)

# Lancer l'application
interface.launch(share=True)